In [ ]:
import os
import math
import random
import pickle
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

DRIVE_DIR = "/content/drive/MyDrive/GPT_boga"
FT_DIR = f"{DRIVE_DIR}/finetuning"

BEST_PRETRAINED = f"{DRIVE_DIR}/best_pretrained_model.pt"
print(device)

cuda


In [ ]:
vocab_size = 4097
padded_vocab_size = 4160

block_size = 256
n_embd = 512
n_head = 8
n_layer = 8
dropout = 0.05

In [ ]:
class Head(nn.Module):

  def __init__(self, head_size):
    super().__init__()
    self.key = nn.Linear(n_embd, head_size, bias= False)
    self.query = nn.Linear(n_embd, head_size, bias = False)
    self.value = nn.Linear(n_embd, head_size, bias = False)
    self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    B, T, C = x.shape
    k = self.key(x)
    q = self.query(x)
    v = self.value(x)
    out = F.scaled_dot_product_attention(q, k, v, attn_mask = None, dropout_p = dropout if self.training else 0.0, is_causal = True)
    return out

In [ ]:
class MultiHeadAttention(nn.Module):

  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
    self.proj = nn.Linear(num_heads * head_size, n_embd)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    out = torch.cat([h(x) for h in self.heads], dim = -1)
    out = self.dropout(self.proj(out))
    return out

In [ ]:
class FeedForward(nn.Module):

  def __init__(self, n_embd):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embd, n_embd*4),
        nn.ReLU(),
        nn.Linear(n_embd*4, n_embd),
        nn.Dropout(dropout),
    )

  def forward(self, x):
    return self.net(x)

In [ ]:
class Block(nn.Module):

  def __init__(self, n_embd, n_head):
    super().__init__()
    head_size = n_embd // n_head
    self.sa = MultiHeadAttention(n_head, head_size)
    self.ffwd = FeedForward(n_embd)
    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self, x):
    x = x + self.sa(self.ln1(x))
    x = x + self.ffwd(self.ln2(x))
    return x

In [ ]:
class GPTLanguageModel(nn.Module):

  def __init__(self):
    super().__init__()

    #self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
    self.token_embedding_table = nn.Embedding(padded_vocab_size, n_embd)
    self.position_embedding_table = nn.Embedding(block_size, n_embd)
    self.blocks = nn.Sequential(*[Block(n_embd, n_head = n_head) for _ in range(n_layer)])
    self.ln_f = nn.LayerNorm(n_embd) # final LN
    self.lm_head = nn.Linear(n_embd, padded_vocab_size)

    self.token_embedding_table.weight = self.lm_head.weight   # <- weight tying    ### new ***

    self.apply(self._init_weights)

  def _init_weights(self, module):
    if isinstance(module, nn.Linear):
      torch.nn.init.normal_(module.weight, mean = 0.0, std = 0.02)
      if module.bias is not None:
        torch.nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
      torch.nn.init.normal_(module.weight, mean = 0.0, std = 0.02)

  def forward(self, idx, targets = None):
    B, T = idx.shape

    tok_emb = self.token_embedding_table(idx) # (B, T, C)
    pos_emb = self.position_embedding_table(torch.arange(T, device = idx.device)) # (T, C)
    x = tok_emb + pos_emb # (B, T, C)
    x = self.blocks(x) # (B, T, C)
    x = self.ln_f(x) # (B, T, C)
    logits = self.lm_head(x) # (B, T, vocab_size)

    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets, ignore_index=-100)

    return logits, loss

  def generate(self, idx, max_new_tokens):
    for _ in range(max_new_tokens):
      idx_cond = idx[:, -block_size:]
      logits, loss = self(idx_cond)
      logits = logits[:, -1, :vocab_size]
      probs = F.softmax(logits, dim = -1)
      idx_next = torch.multinomial(probs, num_samples = 1)
      idx = torch.cat((idx, idx_next), dim = 1)
    return idx


In [ ]:
raw_model = GPTLanguageModel().to(device)

checkpoint = torch.load(BEST_PRETRAINED, map_location = device, weights_only = True)

raw_model.load_state_dict(checkpoint["model"])

model = torch.compile(
    raw_model,
    mode="reduce-overhead"
)

print("Loaded pretrained step: ", checkpoint.get("step"))
print("Pretrained val loss: ", checkpoint.get("best_val"))

Loaded pretrained step:  34800
Pretrained val loss:  None


In [ ]:
train_ids = np.memmap(f"{FT_DIR}/train_ids.bin", dtype=np.uint16, mode="r")
train_mask = np.memmap(f"{FT_DIR}/train_mask.bin", dtype=np.uint8, mode="r")
train_offsets = np.load(f"{FT_DIR}/train_offsets.npy")
val_ids = np.memmap(f"{FT_DIR}/val_ids.bin", dtype=np.uint16, mode="r")
val_mask = np.memmap(f"{FT_DIR}/val_mask.bin", dtype=np.uint8, mode="r")
val_offsets = np.load(f"{FT_DIR}/val_offsets.npy")

print("train examples:", len(train_offsets) - 1)
print("val examples:", len(val_offsets) - 1)

train examples: 49020
val examples: 3072


In [ ]:
IGNORE_INDEX = -100
batch_size = 32

In [ ]:
def get_batch(split):

  if split == "train":
    ids_data = train_ids
    mask_data = train_mask
    offsets = train_offsets
  else:
    ids_data = val_ids
    mask_data = val_mask
    offsets = val_offsets

  n_examples = len(offsets) - 1
  indices = torch.randint(0, n_examples, (batch_size,))

  x_batch = []
  y_batch = []

  for idx in indices:
    start = offsets[idx]
    end = offsets[idx+1]

    ids = np.asarray(ids_data[start:end], dtype = np.int64)
    mask = np.asarray(mask_data[start:end], dtype = np.int8)

    x = ids[:-1]
    y = ids[1:]

    target_mask = mask[1:]

    y = y.copy()
    y[target_mask == 0] = IGNORE_INDEX

    pad_len = block_size - len(x)

    if pad_len > 0:
      x = np.pad(x, (0, pad_len), constant_values = 0)
      y = np.pad(y, (0, pad_len), constant_values = IGNORE_INDEX)

    x_batch.append(torch.from_numpy(x))
    y_batch.append(torch.from_numpy(y))

  x = torch.stack(x_batch).to(device)
  y = torch.stack(y_batch).to(device)

  return x, y

In [ ]:
#loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T), ignore_index = -100)

In [ ]:
#SANITY CHECK-----------------------


xb, yb = get_batch("train")

print(xb.shape)
print(yb.shape)

print(
    "supervised targets:",
    (yb != IGNORE_INDEX).sum().item()
)

torch.Size([32, 256])
torch.Size([32, 256])
supervised targets: 4960


In [ ]:
#SANITY CHECK-----------------------

xb, yb = get_batch("train")

xb = xb.to(device)
yb = yb.to(device)

with torch.no_grad():
    logits, loss = raw_model(xb, yb)

print("loss:", loss.item())
print("finite:", torch.isfinite(loss))

loss: 1.3157349824905396
finite: tensor(True, device='cuda:0')


In [ ]:
import time
import math
import os

batch_size = 32
learning_rate = 3e-5

max_iters = 3200
eval_interval = 100
eval_iters = 50

warmup_iters = 100
min_lr = 3e-6

grad_clip = 1.0

SAVE_DIR = f"{DRIVE_DIR}/sft_4"
os.makedirs(SAVE_DIR, exist_ok=True)

BEST_SFT_PATH = f"{SAVE_DIR}/best_sft_model_4.pt"
FINAL_SFT_PATH = f"{SAVE_DIR}/final_sft_model_4.pt"


# ---------------------------------------
# Model


raw_model = raw_model.to(device)

model = torch.compile(
    raw_model,
    mode="reduce-overhead"
)


# ----------------------------------------------
# Optimizer + scaler

optimizer = torch.optim.AdamW(raw_model.parameters(), lr=learning_rate, fused=(device == "cuda")
)

scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))


# -----------------------
# LR scheduler

def get_lr(it):

    if it < warmup_iters:
        return learning_rate * (it + 1) / warmup_iters

    if it >= max_iters:
        return min_lr

    progress = ((it - warmup_iters) / (max_iters - warmup_iters))

    coeff = 0.5 * (1.0 + math.cos(math.pi * progress))

    return min_lr + coeff * (learning_rate - min_lr)


# ------------------------------------------------
# Validation

@torch.no_grad()
def estimate_loss():

    raw_model.eval()

    losses = []

    for _ in range(eval_iters):

        xb, yb = get_batch("val")

        with torch.autocast(device_type="cuda",
            dtype=torch.float16,
            enabled=(device == "cuda")
        ):
            _, loss = model(xb, yb)

        losses.append(loss.item())

    raw_model.train()

    return sum(losses) / len(losses)


# --------------------------------------
# Training

best_val = float("inf")

raw_model.train()

t0 = time.time()

for step in range(max_iters):

    # LR
    lr = get_lr(step)

    for param_group in optimizer.param_groups:
        param_group["lr"] = lr

    # Batch
    xb, yb = get_batch("train")

    optimizer.zero_grad(set_to_none=True)

    # Forward
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
        _, loss = model(xb, yb)

    train_loss = loss.detach().item()
    # Backward
    scaler.scale(loss).backward()

    # Required before gradient clipping
    scaler.unscale_(optimizer)

    torch.nn.utils.clip_grad_norm_(raw_model.parameters(), grad_clip)

    scaler.step(optimizer)
    scaler.update()


    # Evaluation
    if (
        step % eval_interval == 0 or step == max_iters - 1
    ):

        val_loss = estimate_loss()

        elapsed = time.time() - t0

        print(
            f"step {step:4d} | "
            f"train {train_loss:.4f} | "
            f"val {val_loss:.4f} | "
            f"lr {lr:.2e} | "
            f"time {elapsed:.1f}s"
        )

        # Save best
        if val_loss < best_val:

            best_val = val_loss

            torch.save(
                {
                    "model": raw_model.state_dict(),
                    "step": step,
                    "val_loss": val_loss,
                    "learning_rate": lr,
                },
                BEST_SFT_PATH
            )

            print(
                f"  saved best model "
                f"(val={val_loss:.4f})"
            )


# ----------------------------------------------
# Final model


torch.save(
    {
        "model": raw_model.state_dict(),
        "step": max_iters - 1,
        "val_loss": val_loss,
        "learning_rate": get_lr(max_iters - 1),
    },
    FINAL_SFT_PATH
)

print("\nTraining complete")
print("Best val loss:", best_val)
print("Best model:", BEST_SFT_PATH)
print("Final model:", FINAL_SFT_PATH)

W0924 07:49:22.791000 7382 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


step    0 | train 3.5992 | val 1.3952 | lr 3.00e-07 | time 123.7s
  saved best model (val=1.3952)
step  100 | train 1.6644 | val 1.2787 | lr 3.00e-05 | time 154.4s
  saved best model (val=1.2787)
step  200 | train 1.4796 | val 1.2242 | lr 2.99e-05 | time 178.1s
  saved best model (val=1.2242)
step  300 | train 1.4053 | val 1.2010 | lr 2.97e-05 | time 202.1s
  saved best model (val=1.2010)
step  400 | train 1.4137 | val 1.1997 | lr 2.94e-05 | time 226.7s
  saved best model (val=1.1997)
step  500 | train 1.3369 | val 1.1968 | lr 2.89e-05 | time 251.0s
  saved best model (val=1.1968)
step  600 | train 1.2845 | val 1.1854 | lr 2.83e-05 | time 275.1s
  saved best model (val=1.1854)
step  700 | train 1.3184 | val 1.1904 | lr 2.76e-05 | time 299.4s
step  800 | train 1.2970 | val 1.1716 | lr 2.67e-05 | time 323.2s
  saved best model (val=1.1716)
step  900 | train 1.2265 | val 1.1721 | lr 2.58e-05 | time 347.4s
step 1000 | train 1.4613 | val 1.1816 | lr 2.48e-05 | time 371.1s
step 1100 | train 

In [ ]:

def build_vocab(merges):
    vocab = {i: bytes([i]) for i in range(256)}

    for (a, b), idx in merges.items():
        vocab[idx] = vocab[a] + vocab[b]

    return vocab

vocab = build_vocab(merges)

In [ ]:
import regex as re
import numpy as np
import pickle

pat = re.compile(
    r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
)

def merge(ids, pair, idx):
    out, i = [], 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
            out.append(idx)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return out


def encode(text, merges, cache):
    out = []
    for w in pat.findall(text):
        if w not in cache:
            ids = list(w.encode("utf-8"))
            while len(ids) >= 2:
                pairs = set(zip(ids, ids[1:]))
                pair = min(
                    pairs,
                    key=lambda p: merges.get(p, float("inf"))
                )
                if pair not in merges:
                    break
                ids = merge(ids, pair, merges[pair])
            cache[w] = ids
        out.extend(cache[w])
    return out

def decode(ids, vocab):
    return b"".join(vocab[i] for i in ids).decode("utf-8", errors="replace")

In [ ]:
BEST_SFT_PATH = f"{DRIVE_DIR}/sft_2/best_sft_model_2.pt"

sft_model = GPTLanguageModel().to(device)

checkpoint = torch.load(
    BEST_SFT_PATH,
    map_location=device,
    weights_only=True
)

sft_model.load_state_dict(checkpoint["model"])
sft_model.eval()

print("Best SFT step:", checkpoint["step"])
print("Best val loss:", checkpoint["val_loss"])

Best SFT step: 3000
Best val loss: 1.0995801019668578


In [ ]:
prompt = """Words: dog, garden, happy
Features: Dialogue
Summary: A little girl finds a lost dog in her garden and helps it return home.
Story:
"""

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/GPT_boga"

merges = pickle.load(
    open(f"{DRIVE_DIR}/merges.pkl", "rb")
)

EOT = 4096
cache = {}

prompt_ids = encode(
    prompt,
    merges,
    cache
)

x = torch.tensor(
    [prompt_ids],
    dtype=torch.long,
    device=device
)

In [ ]:
@torch.inference_mode()
def generate_story(
    model,
    idx,
    max_new_tokens=200,
    temperature=0.8
):

    for _ in range(max_new_tokens):

        idx_cond = idx[:, -block_size:]

        logits, _ = model(idx_cond)

        logits = logits[:, -1, :vocab_size]

        logits = logits / temperature

        probs = F.softmax(logits, dim=-1)

        idx_next = torch.multinomial(
            probs,
            num_samples=1
        )

        idx = torch.cat(
            (idx, idx_next),
            dim=1
        )

        # stop at EOT
        if idx_next.item() == EOT:
            break

    return idx

In [ ]:
out = generate_story(
    sft_model,
    x,
    max_new_tokens=200,
    temperature=0.8
)[0].tolist()

In [ ]:
generated_ids = out[len(prompt_ids):]

if EOT in generated_ids:
    generated_ids = generated_ids[
        :generated_ids.index(EOT)
    ]

story = decode(
    generated_ids,
    vocab
)

print(prompt)
print(story)

Words: dog, garden, happy
Features: Dialogue
Summary: A little girl finds a lost dog in her garden and helps it return home.
Story:

Once upon a time there was a little girl. She loved to play in her garden. In her garden, she liked to play with her toys. One day, she found a cute little dog. The dog was lost and very sad.
The little girl wanted to keep the dog safe. She ran to her house and got a ball. She showed the dog to her mom. Her mom said the dog needed a home. The little girl and her mom put the dog in a box.
The little girl took the box to her room. She gave the dog a name. She called him "Coco". The dog wagged his tail. The little girl was happy and took the box to her room. She gave Coco a hug. Coco was happy too. They played all day in her garden.


In [ ]:
prompts = [
"""Words: dog, garden, happy
Summary: A girl finds a lost dog and helps it return home.
Story:
""",

"""Words: spaceship, moon, brave
Features: Dialogue
Summary: Two children build a pretend spaceship and imagine visiting the moon.
Story:
""",

"""Words: rain, umbrella, friend
Features: Dialogue
Summary: A boy shares his umbrella with a friend during a storm.
Story:
""",
]

In [ ]:
# ------------------------------------------------------------
# Compare pretrained vs SFT on the exact same prompt
# ------------------------------------------------------------

PRETRAINED_PATH = f"{DRIVE_DIR}/best_pretrained_model.pt"
SFT_PATH = f"{DRIVE_DIR}/sft_2/best_sft_model_2.pt"

prompt = """Words: snow, meadows, cold
Features: Dialogue
Summary: A far land having meadows, got snow in winter and was cold.
Story:
"""

# ------------------------------------------------------------
# Helper: load checkpoint into a fresh model
# ------------------------------------------------------------

def load_model(path):

    model = GPTLanguageModel().to(device)

    ckpt = torch.load(
        path,
        map_location=device,
        weights_only=True
    )

    model.load_state_dict(ckpt["model"])
    model.eval()

    return model, ckpt


# ------------------------------------------------------------
# Generation helper
# ------------------------------------------------------------

@torch.inference_mode()
def generate_story(
    model,
    prompt,
    max_new_tokens=200,
    temperature=0.8,
    seed=42
):

    # same random seed for a fairer comparison
    torch.manual_seed(seed)

    if device == "cuda":
        torch.cuda.manual_seed(seed)

    cache = {}

    prompt_ids = encode(
        prompt,
        merges,
        cache
    )

    x = torch.tensor(
        [prompt_ids],
        dtype=torch.long,
        device=device
    )

    for _ in range(max_new_tokens):

        idx_cond = x[:, -block_size:]

        logits, _ = model(idx_cond)

        # only real tokenizer vocabulary
        logits = logits[:, -1, :vocab_size]

        logits = logits / temperature

        probs = F.softmax(
            logits,
            dim=-1
        )

        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        x = torch.cat(
            (x, next_token),
            dim=1
        )

        # stop if model generates EOT
        if next_token.item() == EOT:
            break

    generated_ids = x[0].tolist()[len(prompt_ids):]

    # remove EOT before decode
    if EOT in generated_ids:
        generated_ids = generated_ids[
            :generated_ids.index(EOT)
        ]

    return decode(
        generated_ids,
        vocab
    )


# ------------------------------------------------------------
# Load both models
# ------------------------------------------------------------

pretrained_model, pretrained_ckpt = load_model(
    PRETRAINED_PATH
)

sft_model, sft_ckpt = load_model(
    SFT_PATH
)


# ------------------------------------------------------------
# Generate with identical settings
# ------------------------------------------------------------

pretrained_story = generate_story(
    pretrained_model,
    prompt,
    max_new_tokens=200,
    temperature=0.8,
    seed=42
)

sft_story = generate_story(
    sft_model,
    prompt,
    max_new_tokens=200,
    temperature=0.8,
    seed=42
)


# ------------------------------------------------------------
# Display comparison
# ------------------------------------------------------------

print("=" * 100)
print("PROMPT")
print("=" * 100)
print(prompt)

print("\n" + "=" * 100)
print("PRETRAINED MODEL")
print("=" * 100)
print(pretrained_story)

print("\n" + "=" * 100)
print("SFT MODEL")
print("=" * 100)
print(sft_story)

PROMPT
Words: snow, meadows, cold
Features: Dialogue
Summary: A far land having meadows, got snow in winter and was cold.
Story:


PRETRAINED MODEL
Greature: 
"Yum! Let's play in the meadow tomorrow!" 
Bunny: 
Then Bird:
"Let's find out who left the meadow hop the next day." 
Summer Hippo:
"Choooshy! Let's believe."

The two friends ran back home, thinking of the special moment that they had just made.

SFT MODEL
Once upon a time there was a far land. It was full of cold things. It was time for the cold it snowed.
One day, the sky began to turn grey. The snowflakes started to fall from the sky. The snowflakes and trees were frozen. It was so cold that the middle of the snow was icy.
But then, the snowflakes began to fall. Suddenly, the snowflakes started to melt. They splashed and splashed until they were gone.
The night sky was dark, and the snow was freezing. The warm things in the meadow scattered.
The sun and the melting snow were gone and the sky was blue again. But the fields wer

In [ ]:
# ------------------------------------------------------------
# Compare 3 SFT checkpoints on the same prompt suite
# ------------------------------------------------------------

import torch
import torch.nn.functional as F

SFT_PATHS = {
    "dropout_0.0": f"{DRIVE_DIR}/sft_2/best_sft_model_2.pt",
    "dropout_0.1": f"{DRIVE_DIR}/sft_3/best_sft_model_3.pt",
    "dropout_0.05": f"{DRIVE_DIR}/sft_4/best_sft_model_4.pt",
}

prompts = [
"""Words: dog, garden, happy
Features: Dialogue
Summary: A little girl finds a lost dog in her garden and helps it return home.
Story:
""",

"""Words: rainbow, mountain, happy
Features: Dialogue
Summary: A good traveller finds a rainbow in the mountains and gets happy.
Story:
""",

"""Words: snow, meadows, cold
Features: Dialogue
Summary: A far land having meadows got snow in winter and was cold.
Story:
""",

"""Words: cat, window, afraid
Features: Dialogue
Summary: A cat gets stuck near a window and a child helps it feel safe.
Story:
""",

"""Words: river, boat, brave
Features: Dialogue
Summary: Two children cross a river in a small boat and learn to be brave.
Story:
""",

"""Words: apple, tree, kind
Features: Dialogue
Summary: A child shares apples from a tree with a hungry friend.
Story:
""",

"""Words: rain, umbrella, friend
Features: Dialogue
Summary: A boy shares his umbrella with a friend during heavy rain.
Story:
""",

"""Words: moon, spaceship, excited
Features: Dialogue
Summary: Two children pretend to fly a spaceship to the moon.
Story:
""",

"""Words: puppy, street, lost
Features: Dialogue
Summary: A lost puppy is found on the street and returned to its family.
Story:
""",

"""Words: beach, shell, beautiful
Features: Dialogue
Summary: A girl finds a beautiful shell on the beach and shows it to her brother.
Story:
""",

"""Words: forest, owl, dark
Features: Dialogue
Summary: A child walks through a dark forest and meets a friendly owl.
Story:
""",

"""Words: cake, birthday, surprise
Features: Dialogue
Summary: A family prepares a surprise birthday cake for a little boy.
Story:
""",

"""Words: school, pencil, helpful
Features: Dialogue
Summary: A student lends a pencil to a classmate who forgot theirs.
Story:
""",

"""Words: pond, frog, green
Features: Dialogue
Summary: Two friends find a green frog near a pond and watch it jump.
Story:
""",

"""Words: kite, wind, high
Features: Dialogue
Summary: A child flies a kite high in the wind and feels proud.
Story:
""",

"""Words: rabbit, carrot, hungry
Features: Dialogue
Summary: A hungry rabbit searches for a carrot and finally finds one.
Story:
""",

"""Words: train, station, excited
Features: Dialogue
Summary: A child visits a train station for the first time and gets excited.
Story:
""",

"""Words: stars, night, wish
Features: Dialogue
Summary: Two children look at the stars at night and make a wish.
Story:
""",

"""Words: toy, broken, repair
Features: Dialogue
Summary: A child breaks a favorite toy and learns how to repair it.
Story:
""",

"""Words: garden, butterfly, gentle
Features: Dialogue
Summary: A child sees a butterfly in the garden and learns to be gentle with it.
Story:
""",
]


def load_checkpoint_model(path):

    model = GPTLanguageModel().to(device)

    ckpt = torch.load(
        path,
        map_location=device,
        weights_only=True
    )

    model.load_state_dict(ckpt["model"])
    model.eval()

    return model, ckpt


@torch.inference_mode()
def generate_story(
    model,
    prompt,
    max_new_tokens=200,
    temperature=0.8,
    seed=42
):

    torch.manual_seed(seed)

    if device == "cuda":
        torch.cuda.manual_seed(seed)

    local_cache = {}

    prompt_ids = encode(
        prompt,
        merges,
        local_cache
    )

    x = torch.tensor(
        [prompt_ids],
        dtype=torch.long,
        device=device
    )

    for _ in range(max_new_tokens):

        idx_cond = x[:, -block_size:]

        logits, _ = model(idx_cond)

        logits = logits[:, -1, :vocab_size]

        logits = logits / temperature

        probs = F.softmax(
            logits,
            dim=-1
        )

        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        x = torch.cat(
            (x, next_token),
            dim=1
        )

        if next_token.item() == EOT:
            break

    generated_ids = x[0].tolist()[len(prompt_ids):]

    if EOT in generated_ids:
        generated_ids = generated_ids[
            :generated_ids.index(EOT)
        ]

    return decode(
        generated_ids,
        vocab
    )


# ------------------------------------------------------------
# Load all 3 models
# ------------------------------------------------------------

models = {}
metadata = {}

for name, path in SFT_PATHS.items():

    model, ckpt = load_checkpoint_model(path)

    models[name] = model
    metadata[name] = ckpt

    print(
        name,
        "| step:",
        ckpt.get("step"),
        "| val:",
        ckpt.get("val_loss")
    )


# ------------------------------------------------------------
# Run evaluation
# ------------------------------------------------------------

results = []

for i, prompt in enumerate(prompts):

    print("\n" + "=" * 120)
    print(f"PROMPT {i + 1}")
    print("=" * 120)
    print(prompt)

    row = {
        "prompt": prompt
    }

    for name, model in models.items():

        story = generate_story(
            model,
            prompt,
            max_new_tokens=200,
            temperature=0.8,
            seed=42
        )

        row[name] = story

        print("\n" + "-" * 120)
        print(name)
        print("-" * 120)
        print(story)

    results.append(row)

dropout_0.0 | step: 3000 | val: 1.0995801019668578
dropout_0.1 | step: 2300 | val: 1.1707087993621825
dropout_0.05 | step: 3000 | val: 1.1336585569381714

PROMPT 1
Words: dog, garden, happy
Features: Dialogue
Summary: A little girl finds a lost dog in her garden and helps it return home.
Story:


------------------------------------------------------------------------------------------------------------------------
dropout_0.0
------------------------------------------------------------------------------------------------------------------------
Once upon a time there was a little girl. She loved to play in her garden. In her garden, she liked to play with her toys. One day, she found a cute little dog. The dog was lost and very sad.
The little girl wanted to keep the dog safe. She ran to her house and got a ball. She showed the dog to her mom. Her mom said the dog needed a home. The little girl and her mom put the dog in a box.
The little girl took the box to her room. She gave the do

# Fine Tuning the pretrained_best using LoRA

In [ ]:
#BEST_PRETRAINED = f"{DRIVE_DIR}/best_pretrained_model.pt"
BEST_PRETRAINED = f"{DRIVE_DIR}/sft_2/best_sft_model_2.pt"

raw_model = GPTLanguageModel().to(device)

checkpoint = torch.load(
    BEST_PRETRAINED,
    map_location=device,
    weights_only=True
)

raw_model.load_state_dict(checkpoint["model"])

print("Loaded pretrained step:", checkpoint.get("step"))
print("Pretrained val loss:", checkpoint.get("best_val"))

Loaded pretrained step: 3000
Pretrained val loss: None


In [ ]:
# Define LoRA

import math

class LoRALinear(nn.Module):

    def __init__(self, base_layer, r=8, alpha=16, dropout=0.0):

        super().__init__()
        self.base = base_layer
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r

        self.lora_dropout = nn.Dropout(dropout)

        self.lora_A = nn.Linear(base_layer.in_features, r, bias=False)

        self.lora_B = nn.Linear(r, base_layer.out_features, bias=False)

        # LoRA init:
        # A random, B zero => initial delta = 0
        nn.init.kaiming_uniform_(self.lora_A.weight,a=math.sqrt(5))

        nn.init.zeros_(self.lora_B.weight)

        # Freeze original layer
        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self, x):

        base_out = self.base(x)

        lora_out = self.lora_B(
            self.lora_A(
                self.lora_dropout(x)
            )
        )

        return (base_out + self.scaling * lora_out)

In [ ]:
for param in raw_model.parameters():
    param.requires_grad = False

In [ ]:
def add_lora(
    module,
    r=8,
    alpha=16,
    dropout=0.0,
    prefix=""
):

    for name, child in list(module.named_children()):

        full_name = f"{prefix}.{name}" if prefix else name

        # Already wrapped — don't descend into it
        if isinstance(child, LoRALinear):
            continue

        if isinstance(child, nn.Linear):

            if full_name == "lm_head":
                continue

            setattr(module,name,LoRALinear(child, r=r, alpha=alpha, dropout=dropout))

        else:
            add_lora(child, r=r, alpha=alpha, dropout=dropout, prefix=full_name)

In [ ]:
add_lora(
    raw_model,
    r=8,
    alpha=16,
    dropout=0.0
)

raw_model = raw_model.to(device)

In [ ]:
#Verify that LoRA params are trained

total_params = sum(p.numel() for p in raw_model.parameters())

trainable_params = sum(p.numel() for p in raw_model.parameters() if p.requires_grad)

print(f"Total params:     {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

print(
    f"Trainable %: "
    f"{100 * trainable_params / total_params:.3f}%"
)

Total params:     28,750,912
Trainable params: 1,277,952
Trainable %: 4.445%


In [ ]:
for name, p in raw_model.named_parameters():

    if p.requires_grad:
        print(name, p.shape)

blocks.0.sa.heads.0.key.lora_A.weight torch.Size([8, 512])
blocks.0.sa.heads.0.key.lora_B.weight torch.Size([64, 8])
blocks.0.sa.heads.0.query.lora_A.weight torch.Size([8, 512])
blocks.0.sa.heads.0.query.lora_B.weight torch.Size([64, 8])
blocks.0.sa.heads.0.value.lora_A.weight torch.Size([8, 512])
blocks.0.sa.heads.0.value.lora_B.weight torch.Size([64, 8])
blocks.0.sa.heads.1.key.lora_A.weight torch.Size([8, 512])
blocks.0.sa.heads.1.key.lora_B.weight torch.Size([64, 8])
blocks.0.sa.heads.1.query.lora_A.weight torch.Size([8, 512])
blocks.0.sa.heads.1.query.lora_B.weight torch.Size([64, 8])
blocks.0.sa.heads.1.value.lora_A.weight torch.Size([8, 512])
blocks.0.sa.heads.1.value.lora_B.weight torch.Size([64, 8])
blocks.0.sa.heads.2.key.lora_A.weight torch.Size([8, 512])
blocks.0.sa.heads.2.key.lora_B.weight torch.Size([64, 8])
blocks.0.sa.heads.2.query.lora_A.weight torch.Size([8, 512])
blocks.0.sa.heads.2.query.lora_B.weight torch.Size([64, 8])
blocks.0.sa.heads.2.value.lora_A.weight torc

In [ ]:
# check the above op, if seen lora, then proceed

model = torch.compile(raw_model, mode="reduce-overhead")

In [ ]:
learning_rate = 1e-4

max_iters = 3200
warmup_iters = 100
min_lr = 1e-5

eval_interval = 100
eval_iters = 50

grad_clip = 1.0


trainable_parameters = [p for p in raw_model.parameters() if p.requires_grad]

optimizer = torch.optim.AdamW(trainable_parameters, lr=learning_rate, fused=(device == "cuda"))

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(device == "cuda")
)

In [ ]:
def get_lr(it):

    if it < warmup_iters:
        return (
            learning_rate
            * (it + 1)
            / warmup_iters
        )

    if it >= max_iters:
        return min_lr

    progress = (
        (it - warmup_iters)
        / (max_iters - warmup_iters)
    )

    coeff = 0.5 * (
        1.0
        + math.cos(math.pi * progress)
    )

    return (
        min_lr
        + coeff
        * (learning_rate - min_lr)
    )

In [ ]:
@torch.no_grad()
def estimate_loss():

    raw_model.eval()

    losses = []

    for _ in range(eval_iters):
        xb, yb = get_batch("val")
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device == "cuda")
        ):
            _, loss = model(xb, yb)
        losses.append(
            loss.item()
        )
    raw_model.train()
    return (
        sum(losses)
        / len(losses)
    )

In [ ]:
def get_lora_state_dict(model):

    return {
        name: tensor.detach().cpu()
        for name, tensor
        in model.state_dict().items()
        if (
            "lora_A" in name
            or "lora_B" in name
        )
    }

In [ ]:
LORA_DIR = f"{DRIVE_DIR}/lora_sft"
os.makedirs(LORA_DIR, exist_ok=True)

BEST_LORA_PATH = (
    f"{LORA_DIR}/best_lora_sft_adapters.pt"
)

FINAL_LORA_PATH = (
    f"{LORA_DIR}/final_lora_sft_adapters.pt"
)

In [ ]:
import time

best_val = float("inf")

raw_model.train()

t0 = time.time()

for step in range(max_iters):

    lr = get_lr(step)

    for param_group in optimizer.param_groups:
        param_group["lr"] = lr

    xb, yb = get_batch("train")

    optimizer.zero_grad(
        set_to_none=True
    )

    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
        enabled=(device == "cuda")
    ):

        _, loss = model(xb, yb)

    # Store before another compiled forward
    train_loss = loss.detach().item()

    scaler.scale(loss).backward()

    scaler.unscale_(optimizer)

    torch.nn.utils.clip_grad_norm_(
        trainable_parameters,
        grad_clip
    )

    scaler.step(optimizer)
    scaler.update()

    # -----------------------------
    # Evaluation

    if (
        step % eval_interval == 0
        or step == max_iters - 1
    ):

        val_loss = estimate_loss()

        elapsed = time.time() - t0

        print(
            f"step {step:4d} | "
            f"train {train_loss:.4f} | "
            f"val {val_loss:.4f} | "
            f"lr {lr:.2e} | "
            f"time {elapsed:.1f}s"
        )

        if val_loss < best_val:

            best_val = val_loss

            torch.save(
                {
                    "lora": get_lora_state_dict(
                        raw_model
                    ),
                    "step": step,
                    "val_loss": val_loss,
                    "r": 8,
                    "alpha": 16,
                    "lora_dropout": 0.0,
                },
                BEST_LORA_PATH
            )

            print(
                f"  saved best LoRA "
                f"(val={val_loss:.4f})"
            )

step    0 | train 3.2914 | val 1.0990 | lr 1.00e-06 | time 29.1s
  saved best LoRA (val=1.0990)
step  100 | train 2.0824 | val 1.2658 | lr 1.00e-04 | time 55.5s
step  200 | train 1.5578 | val 1.1959 | lr 9.98e-05 | time 80.0s
step  300 | train 1.5576 | val 1.1974 | lr 9.91e-05 | time 104.5s
step  400 | train 1.5286 | val 1.1763 | lr 9.79e-05 | time 129.0s
step  500 | train 1.4064 | val 1.1791 | lr 9.64e-05 | time 153.6s
step  600 | train 1.3145 | val 1.1975 | lr 9.43e-05 | time 178.2s
step  700 | train 1.3959 | val 1.1796 | lr 9.19e-05 | time 202.7s
step  800 | train 1.4284 | val 1.1675 | lr 8.91e-05 | time 227.4s
step  900 | train 1.2483 | val 1.1650 | lr 8.60e-05 | time 251.9s
step 1000 | train 1.4159 | val 1.1771 | lr 8.25e-05 | time 276.4s
step 1100 | train 1.3272 | val 1.1707 | lr 7.88e-05 | time 300.9s
step 1200 | train 1.2709 | val 1.1650 | lr 7.48e-05 | time 325.5s
step 1300 | train 1.3304 | val 1.1715 | lr 7.06e-05 | time 350.1s
step 1400 | train 1.2880 | val 1.1776 | lr 6.63e

In [ ]:
torch.save(
    {
        "lora": get_lora_state_dict(
            raw_model
        ),
        "step": max_iters - 1,
        "val_loss": val_loss,
        "r": 8,
        "alpha": 16,
        "lora_dropout": 0.0,
    },
    FINAL_LORA_PATH
)

print("\nLoRA training complete")
print("Best val loss:", best_val)
print("Best LoRA:", BEST_LORA_PATH)
print("Final LoRA:", FINAL_LORA_PATH)


LoRA training complete
Best val loss: 1.098964067697525
Best LoRA: /content/drive/MyDrive/GPT_boga/lora_sft/best_lora_sft_adapters.pt
Final LoRA: /content/drive/MyDrive/GPT_boga/lora_sft/final_lora_sft_adapters.pt


In [ ]:
# ------------------------------------------------------------
# Compare pretrained vs SFT on the exact same prompt
# ------------------------------------------------------------

PRETRAINED_PATH = f"{DRIVE_DIR}/best_pretrained_model.pt"
SFT_PATH = f"{DRIVE_DIR}/sft_2/best_sft_model_2.pt"
LORA_PATH = f"{DRIVE_DIR}/lora_sft/final_lora_sft_adapters.pt"

prompt = """Words: snow, meadows, cold
Features: Dialogue
Summary: A far land having meadows, got snow in winter and was cold.
Story:
"""

# ------------------------------------------------------------
# Helper: load checkpoint into a fresh model
# ------------------------------------------------------------

def load_model(path):

    model = GPTLanguageModel().to(device)

    ckpt = torch.load(
        path,
        map_location=device,
        weights_only=True
    )

    model.load_state_dict(ckpt["model"])
    model.eval()

    return model, ckpt

def load_lora_model(
    pretrained_path,
    lora_path,
    r=8,
    alpha=16,
    dropout=0.0
):

    model = GPTLanguageModel().to(device)

    # load original pretrained base
    ckpt = torch.load(
        pretrained_path,
        map_location=device,
        weights_only=True
    )

    model.load_state_dict(ckpt["model"])

    # freeze base
    for p in model.parameters():
        p.requires_grad = False

    # recreate exact LoRA structure
    add_lora(
        model,
        r=r,
        alpha=alpha,
        dropout=dropout
    )

    model = model.to(device)

    # load trained LoRA adapters
    lora_ckpt = torch.load(
        lora_path,
        map_location=device,
        weights_only=True
    )

    missing, unexpected = model.load_state_dict(
        lora_ckpt["lora"],
        strict=False
    )

    model.eval()

    print(
        "Loaded LoRA step:",
        lora_ckpt.get("step")
    )

    print(
        "LoRA val loss:",
        lora_ckpt.get("val_loss")
    )

    return model, lora_ckpt

# ------------------------------------------------------------
# Generation helper
# ------------------------------------------------------------

@torch.inference_mode()
def generate_story(
    model,
    prompt,
    max_new_tokens=200,
    temperature=0.8,
    seed=42
):

    # same random seed for a fairer comparison
    torch.manual_seed(seed)

    if device == "cuda":
        torch.cuda.manual_seed(seed)

    cache = {}

    prompt_ids = encode(
        prompt,
        merges,
        cache
    )

    x = torch.tensor(
        [prompt_ids],
        dtype=torch.long,
        device=device
    )

    for _ in range(max_new_tokens):

        idx_cond = x[:, -block_size:]

        logits, _ = model(idx_cond)

        # only real tokenizer vocabulary
        logits = logits[:, -1, :vocab_size]

        logits = logits / temperature

        probs = F.softmax(
            logits,
            dim=-1
        )

        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        x = torch.cat(
            (x, next_token),
            dim=1
        )

        # stop if model generates EOT
        if next_token.item() == EOT:
            break

    generated_ids = x[0].tolist()[len(prompt_ids):]

    # remove EOT before decode
    if EOT in generated_ids:
        generated_ids = generated_ids[
            :generated_ids.index(EOT)
        ]

    return decode(
        generated_ids,
        vocab
    )


# ------------------------------------------------------------
# Load both models
# ------------------------------------------------------------

pretrained_model, pretrained_ckpt = load_model(
    PRETRAINED_PATH
)

sft_model, sft_ckpt = load_model(
    SFT_PATH
)

lora_model, lora_ckpt = load_lora_model(
    PRETRAINED_PATH,
    LORA_PATH,
    r=8,
    alpha=16,
    dropout=0.0
)

# ------------------------------------------------------------
# Generate with identical settings
# ------------------------------------------------------------

pretrained_story = generate_story(
    pretrained_model,
    prompt,
    max_new_tokens=200,
    temperature=0.8,
    seed=42
)

sft_story = generate_story(
    sft_model,
    prompt,
    max_new_tokens=200,
    temperature=0.8,
    seed=42
)

lora_story = generate_story(
    lora_model,
    prompt,
    max_new_tokens=200,
    temperature=0.8,
    seed=42
)

# ------------------------------------------------------------
# Display comparison
# ------------------------------------------------------------

print("=" * 100)
print("PROMPT")
print("=" * 100)
print(prompt)

print("\n" + "=" * 100)
print("PRETRAINED MODEL")
print("=" * 100)
print(pretrained_story)

print("\n" + "=" * 100)
print("SFT MODEL")
print("=" * 100)
print(sft_story)

print("\n" + "=" * 100)
print("LoRA MODEL")
print("=" * 100)
print(lora_story)

Loaded LoRA step: 3199
LoRA val loss: 1.1648029601573944
PROMPT
Words: snow, meadows, cold
Features: Dialogue
Summary: A far land having meadows, got snow in winter and was cold.
Story:


PRETRAINED MODEL
Greature: 
"Yum! Let's play in the meadow tomorrow!" 
Bunny: 
Then Bird:
"Let's find out who left the meadow hop the next day." 
Summer Hippo:
"Choooshy! Let's believe."

The two friends ran back home, thinking of the special moment that they had just made.

SFT MODEL
Once upon a time there was a far land. It was full of cold things. It was time for the cold it snowed.
One day, the sky began to turn grey. The snowflakes started to fall from the sky. The snowflakes and trees were frozen. It was so cold that the middle of the snow was icy.
But then, the snowflakes began to fall. Suddenly, the snowflakes started to melt. They splashed and splashed until they were gone.
The night sky was dark, and the snow was freezing. The warm things in the meadow scattered.
The sun and the melting snow